In [0]:
-- neue script von Claude_08.092026

-- ============================================================
-- Script : 05_load_facts_northriver (final, bereinigt)
-- Purpose: Idempotenter Load von fact_order_items.
--          Kein store_sk (stg_orders/stg_order_items haben keine
--          store_id), kein SCD2-Point-in-time-Join (dim_customer/
--          dim_product sind Full-Overwrite ohne echte Historie,
--          daher genügt ein einfacher Join über die Business-Keys).
-- Depends: northriver_staging (stg_order_items, stg_orders),
--          northriver_warehouse (dim_customer, dim_product, dim_channel)
-- Outputs: northriver_warehouse.fact_order_items
-- ============================================================

MERGE INTO workspace.northriver_warehouse.fact_order_items AS target
USING (
    SELECT
        oi.order_item_id,
        oi.order_id,
        dc.customer_sk,
        dp.product_sk,
        dch.channel_sk,
        CAST(date_format(o.order_date, 'yyyyMMdd') AS INT) AS date_sk,
        oi.quantity,
        oi.unit_price,
        oi.quantity * oi.unit_price AS revenue
    FROM workspace.northriver_staging.stg_order_items oi
    JOIN workspace.northriver_staging.stg_orders      o   ON oi.order_id  = o.order_id
    JOIN workspace.northriver_warehouse.dim_customer  dc  ON o.customer_id = dc.customer_id
    JOIN workspace.northriver_warehouse.dim_product   dp  ON oi.product_id = dp.product_id
    JOIN workspace.northriver_warehouse.dim_channel   dch ON o.channel     = dch.channel_name
) AS source
ON target.order_item_id = source.order_item_id
WHEN MATCHED AND (
        target.customer_sk <> source.customer_sk
     OR target.product_sk  <> source.product_sk
     OR target.channel_sk  <> source.channel_sk
     OR target.date_sk     <> source.date_sk
     OR target.revenue     <> source.revenue
     OR target.revenue IS NULL
    ) THEN
    UPDATE SET
        target.customer_sk = source.customer_sk,
        target.product_sk  = source.product_sk,
        target.channel_sk  = source.channel_sk,
        target.date_sk     = source.date_sk,
        target.quantity    = source.quantity,
        target.unit_price  = source.unit_price,
        target.revenue     = source.revenue
WHEN NOT MATCHED THEN
    INSERT (order_item_id, order_id, customer_sk, product_sk, channel_sk,
            date_sk, quantity, unit_price, revenue)
    VALUES (source.order_item_id, source.order_id, source.customer_sk, source.product_sk,
            source.channel_sk, source.date_sk, source.quantity, source.unit_price, source.revenue);

-- Validieren
SELECT COUNT(*) AS fact_rows FROM workspace.northriver_warehouse.fact_order_items;
-- Erwartet: 32